# Společný model Random Forest pro úlohy XAI

Jeden model pro všechny úlohy druhé části (PFI, ALE, SHAP). Kontroly správnosti modelu jsou
v samostatném notebooku [`model_checks.ipynb`](model_checks.ipynb), aby načtení modelu zůstalo rychlé.

## Jak model použít ve své úloze

1. Notebook úlohy ulož do vlastní složky přímo v `analyses/xai/`, např. `analyses/xai/shap/moje_uloha.ipynb`.
   Relativní cesty (`../models/...`, `../../../data/...`) fungují jen v této hloubce.
2. Na začátku notebooku spusť:

   ```python
   %run ../models/rf_model.ipynb
   ```

   Celý tento notebook se provede (asi 20 s; model se pokaždé deterministicky přetrénuje, nic se neukládá
   do souboru) a vypíše kvalitu modelu.

**Dostupné proměnné** (neměň je, ostatní úlohy počítají se stejným modelem):

| proměnná | obsah |
|---|---|
| `pipeline` | předzpracování + model, vstup = původní atributy (PFI, ALE) |
| `pipeline.named_steps["preprocess"]` | `ColumnTransformer` s `OrdinalEncoder` (SHAP) |
| `pipeline.named_steps["model"]` | natrénovaný `RandomForestClassifier` (SHAP) |
| `X_train`, `X_test`, `y_train`, `y_test` | rozdělení dat (původní atributy, index = řádek v `df`) |
| `df` | připravená data: 22 atributů + `is_canceled` |
| `FEATURES`, `CATEGORICAL`, `NUMERIC`, `TARGET`, `SEED` | seznamy atributů, název cíle, seed = 42 |
| `group_ids` | ID skupiny shodných rezervací pro každý řádek `df` (použito při rozdělení) |

**Poznámky k použití:**

- Zakódovaná data pro SHAP s názvy sloupců:
  `pd.DataFrame(pipeline.named_steps["preprocess"].transform(X_test), columns=FEATURES, index=X_test.index)`.
  Pořadí sloupců po kódování je stejné jako `FEATURES`.
- Kódy kategorií: `pipeline.named_steps["preprocess"].named_transformers_["cat"].categories_`
  (seznam polí ve stejném pořadí jako `CATEGORICAL`; kód = pozice v poli).
  Kategorie, které v trénovacích datech nejsou (v testu 17 rezervací s neznámým `agent`, 30 s neznámou
  `company`), mají kód −1.
- `agent` a `company` jsou ID s libovolným pořadím kódů – ALE pro ně nemá smysl.
  `arrival_date_month` je číslo 1–12, prosinec a leden tedy nejsou „sousedé“.

## Data a atributy

**Data:** `data/processed/hotel_bookings_clean.csv` (118 558 rezervací, sdílená data z první části).
Cíl: `is_canceled`.

**Atributy modelu (22):** `hotel`, `lead_time`, `arrival_date_month` (číslo 1–12),
`stays_in_weekend_nights`, `stays_in_week_nights`, `adults`, `children`, `babies`, `meal`,
`market_segment`, `distribution_channel`, `is_repeated_guest`, `previous_cancellations`,
`previous_bookings_not_canceled`, `reserved_room_type`, `deposit_type`, `agent`, `company`,
`days_in_waiting_list`, `customer_type`, `adr`, `total_of_special_requests`.

**Vyloučené atributy.** Atribut je přípustný jen tehdy, když je jeho hodnota známá v okamžiku rezervace.

| atribut | důvod |
|---|---|
| `reservation_status`, `reservation_status_date` | jiné zakódování cíle; odstraněno už při přípravě dat |
| `assigned_room_type` | pokoj se přiděluje až při příjezdu (jiný než rezervovaný pokoj: 5,5 % zrušení vs. 41,7 %) |
| `booking_changes` | vzniká po rezervaci a přibývá s časem: zrušené rezervace mají změnu tím častěji, čím později byly zrušeny (4 % → 13 %, viz `model_checks.ipynb`) |
| `required_car_parking_spaces` | při požadavku na parkování je 0 % zrušení, údaj se zjevně zapisuje při příjezdu (stejně jako v 4ft úloze) |
| `country` | **nejisté:** formálně známá při rezervaci, ale u zrušených rezervací je podezřele často `PRT` (65 % zrušení v City Hotelu vs. 27 % u ostatních zemí). Hypotéza (daty neověřená): země se upřesňuje až při check-inu. Proto atribut raději vynecháváme |
| `arrival_date_year`, `arrival_date` | časový trend konkrétního období, nezobecňuje se |
| `arrival_date_week_number`, `arrival_date_day_of_month` | duplikují sezónnost měsíce, jen by si s ním dělily důležitost |
| `total_nights`, `total_guests` | přesné součty jiných atributů |

**Ponechané s výhradou** (čísla v `model_checks.ipynb`):

- `previous_cancellations` – známo při rezervaci, ale s **anomálií**: hodnota 1 má 6 044 rezervací
  a 94,5 % z nich je zrušeno; 54 % jsou skupinové rezervace (`Groups`), 58 % má nevratnou zálohu
  a jen 11 % je označeno jako opakovaný host. Spíše jde o artefakt skupinových rezervací než o historii
  hosta – při interpretaci (PFI, ALE, SHAP) je nutné to zmínit.
- `total_of_special_requests` – požadavky může host doplnit i po rezervaci. U zrušených rezervací
  bez zálohy se podíl s požadavkem s dobou do zrušení mírně zvyšuje (32 % → ≈ 40 %), mnohem méně než
  u `booking_changes`; i pozdě zrušené rezervace mají požadavek výrazně méně často (38 %) než
  nezrušené (55 %). Signál tedy není jen důsledkem času, atribut ponecháváme.
- `days_in_waiting_list` – známo při potvrzení rezervace, před příjezdem.
- `adr` – podle popisu datasetu průměrná denní cena = součet ubytovacích transakcí / počet nocí, tedy
  formálně ne nutně cena v okamžiku rezervace. U zrušených rezervací je ale nenulová (nulová cena je vzácná
  v obou třídách: 1,3 % nezrušených, 0,4 % zrušených), takže jde spíše o rezervovanou cenu; přesný okamžik
  záznamu daty ověřit nelze.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

# Same depth from the repo root for every XAI notebook (models/, pfi/, ale/, shap/)
DATA_PATH = "../../../data/processed/hotel_bookings_clean.csv"
SEED = 42
TARGET = "is_canceled"

CATEGORICAL = [
    "hotel", "meal", "market_segment", "distribution_channel", "reserved_room_type",
    "deposit_type", "customer_type", "agent", "company",
]
NUMERIC = [
    "lead_time", "arrival_date_month", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "days_in_waiting_list", "adr",
    "total_of_special_requests",
]
FEATURES = CATEGORICAL + NUMERIC

MONTHS = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

df = pd.read_csv(DATA_PATH, dtype={"agent": str, "company": str})
# agent / company are ids; missing = booked without an agent / not a company booking
df[["agent", "company"]] = df[["agent", "company"]].fillna("none")
df["arrival_date_month"] = df["arrival_date_month"].map(lambda m: MONTHS.index(m) + 1)
df = df[FEATURES + [TARGET]]
print("Bookings:", len(df), "| attributes:", len(FEATURES))

Bookings: 118558 | attributes: 22


## Rozdělení dat

Data obsahují mnoho rezervací se zcela shodnými hodnotami atributů – hlavně skupinové a balíčkové
rezervace (cestovní kancelář rezervuje více pokojů na stejný termín, za stejnou cenu a se stejnou zálohou).
Nejde o chybu dat, proto je v datech ponecháváme. Při náhodném rozdělení by se ale kopie jedné rezervace
dostaly do trénovacích i testovacích dat a model by je v testu „poznal“ místo predikce
(`model_checks.ipynb`, část A: při náhodném rozdělení má 37 % testovacích rezervací identickou kopii
v trénovacích datech a ROC-AUC na nich je 0,981 oproti 0,831 na ostatních).

Proto rozdělujeme **skupinově**: skupinu tvoří rezervace se shodným vektorem všech 22 atributů a celá
skupina jde buď do trénovacích, nebo do testovacích dat. `StratifiedGroupKFold(n_splits=5, shuffle=True,
random_state=42)` se stratifikací podle `hotel × is_canceled`; testovací data = první fold (≈ 20 %).

In [2]:
X, y = df[FEATURES], df[TARGET]

# bookings with an identical feature vector form one group -> never split between train and test
group_ids = pd.factorize(X.astype(str).agg("|".join, axis=1))[0]
strata = df["hotel"] + "_" + y.astype(str)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(cv.split(X, strata, group_ids))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", len(X_train), "| test:", len(X_test), f"({len(X_test) / len(X):.1%})")
print("Feature vectors shared by train and test:", len(set(group_ids[train_idx]) & set(group_ids[test_idx])))
display(pd.DataFrame({
    "train cancel rate": y_train.groupby(X_train["hotel"]).mean(),
    "test cancel rate": y_test.groupby(X_test["hotel"]).mean(),
    "test bookings": X_test["hotel"].value_counts(),
}).round(3))

Train: 94847 | test: 23711 (20.0%)
Feature vectors shared by train and test: 0


,train cancel rate,test cancel rate,test bookings
hotel,,,
City Hotel,0.419,0.419,15778
Resort Hotel,0.280,0.280,7933


## Kódování a model

**Kódování:** kategoriální atributy `OrdinalEncoder` (chybějící `agent` / `company` = kategorie `none`,
neznámá kategorie = −1), číselné beze změny. Random Forest dělí podle prahů a s pořadovými kódy si poradí;
každý atribut zůstává jedním sloupcem, což zjednoduší ALE i SHAP. Metody na původních atributech
(PFI, ALE) pracují se vstupem celé pipeline, tedy ještě před kódováním.

**Model:** `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42)`,
pevné parametry bez ladění na testovacích datech.

In [3]:
preprocess = ColumnTransformer(
    [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL)],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
pipeline = Pipeline([("preprocess", preprocess), ("model", model)])
pipeline.fit(X_train, y_train);

## Kvalita modelu

In [4]:
proba_train = pipeline.predict_proba(X_train)[:, 1]
proba_test = pipeline.predict_proba(X_test)[:, 1]
print(f"ROC-AUC train: {roc_auc_score(y_train, proba_train):.3f}")
print(f"ROC-AUC test:  {roc_auc_score(y_test, proba_test):.3f}")
print(f"Accuracy test: {accuracy_score(y_test, proba_test > 0.5):.3f}")

for hotel in ["City Hotel", "Resort Hotel"]:
    mask = X_test["hotel"] == hotel
    print(f"{hotel}: {mask.sum()} test bookings, ROC-AUC {roc_auc_score(y_test[mask], proba_test[mask]):.3f}")

# keep only the documented variables in the namespace of notebooks that %run this one
del X, y, strata, cv, train_idx, test_idx, preprocess, model, proba_train, proba_test, hotel, mask, MONTHS

ROC-AUC train: 0.956
ROC-AUC test:  0.893
Accuracy test: 0.830
City Hotel: 15778 test bookings, ROC-AUC 0.903
Resort Hotel: 7933 test bookings, ROC-AUC 0.860


ROC-AUC na testovacích datech 0,893 (trénovací 0,956, tj. overfitting kolem 0,06), accuracy 0,830.
City Hotel: 15 778 testovacích rezervací, ROC-AUC 0,903; Resort Hotel: 7 933, ROC-AUC 0,860.
Skupinová křížová validace na trénovacích datech dává 0,882 ± 0,006 (`model_checks.ipynb`, část B),
testovací fold je tedy mírně příznivý; realistická kvalita modelu na nových rezervacích je ≈ 0,88–0,89.

Ve stejné skupině shodných rezervací mají někdy rezervace různý výsledek (677 skupin, 5 211 rezervací) –
takové případy žádný model nerozliší.